# Plant Pathology 2021 — Multi-label CNN (Sigmoid + BCE)

Соревнование: [Plant Pathology 2021 - FGVC8](https://www.kaggle.com/competitions/plant-pathology-2021-fgvc8)

- **Задача:** мультилейбл-классификация (одно изображение → несколько классов)
- **Модель:** CNN (EfficientNet-B0, ImageNet pretrained) + Sigmoid + BCE
- **Метрика:** Mean F1-Score
- **Runtime:** Kaggle GPU T4 ×2, internet disabled

## Данные: предобработка (ресайз)

Исходные изображения соревнования — крупные JPEG (~4000 px, датасет ~16 ГБ).
При обучении их чтение с диска и декодирование становятся **узким местом**: GPU
простаивает, итерация занимает 5–9 с, и 8 эпох не укладываются в лимит GPU 2 ч.

Поэтому изображения **предварительно уменьшены до 512 px** (большая сторона,
сохранён aspect ratio, JPEG quality 90) отдельным ноутбуком
[`converting-dataset-512-identification-apple.ipynb`](https://www.kaggle.com/code/sergeisemenovdev/converting-dataset-512-identification-apple) и сохранены как
Kaggle Dataset.

Тренировочный пайплайн читает **уже уменьшенные** файлы, поэтому декод почти
бесплатный. Путь к подготовленным данным задаётся в `CFG.DATA_DIR`. Проверочная
ячейка после загрузки `train.csv` печатает размер и вес примера — подтверждение,
что ресайз применён.

In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm.auto import tqdm

In [ ]:
class CFG:
    DATA_DIR = Path("/kaggle/input/datasets/sergeisemenovdev/plant-pathology-512/plant-pathology-512")
    TRAIN_CSV = DATA_DIR / "train.csv"
    TRAIN_IMG_DIR = DATA_DIR / "train_images"
    TEST_IMG_DIR = DATA_DIR / "test_images"
    OUTPUT_DIR = Path("/kaggle/working")

    IMG_SIZE = 256  # B0 рассчитан на ~224; 256 — баланс скорость/качество/память
    BATCH_SIZE = 64  # одна GPU; эффективный batch = 64
    NUM_EPOCHS = 8
    LR = 3e-4
    WEIGHT_DECAY = 1e-4
    VAL_RATIO = 0.15
    NUM_WORKERS = 4  # на Kaggle 4 vCPU; декод больших JPEG — узкое место
    THRESHOLD = 0.5
    SEED = 42
    BACKBONE = "efficientnet_b0"  # efficientnet_b0 | resnet50
    PREFERRED_GPUS = 1
    # Kaggle Dataset с .pth (Add Data). Пример slug: torchvision-pretrained-weights
    WEIGHTS_DIR = Path("/kaggle/input/datasets/sergeisemenovdev/torchvision-pretrained-weights")


BACKBONE_WEIGHT_FILES = {
    "efficientnet_b0": "efficientnet_b0_rwightman-7f5810bc.pth",
    "resnet50": "resnet50-0676ba61.pth",
}


def resolve_weights_path(backbone_name: str) -> Path | None:
    filename = BACKBONE_WEIGHT_FILES[backbone_name]
    for directory in (
        CFG.WEIGHTS_DIR,
        CFG.OUTPUT_DIR / "weights",
        Path("/root/.cache/torch/hub/checkpoints"),
        Path.home() / ".cache/torch/hub/checkpoints",
    ):
        path = directory / filename
        if path.exists():
            return path
    return None


def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


def setup_gpus(preferred: int = 1) -> int:
    """Требует минимум 1 GPU; использует ровно preferred (по умолч. 1)."""
    if not torch.cuda.is_available():
        raise RuntimeError(
            "GPU не найден. Включите GPU в Settings ноутбука (T4 x2) и перезапустите сессию."
        )
    n = torch.cuda.device_count()
    if n < 1:
        raise RuntimeError("CUDA доступна, но GPU не обнаружены.")
    use = min(preferred, n)
    if use < n:
        print(f"Доступно GPU: {n}, используем: {use} (по CFG.PREFERRED_GPUS)")
    else:
        print(f"Доступно GPU: {n} (используем все)")
    return use


seed_everything(CFG.SEED)

device = torch.device("cuda")
n_gpus = setup_gpus(CFG.PREFERRED_GPUS)
print(f"Device: {device}")
for i in range(n_gpus):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

In [ ]:
# Проверка, что подключён предобработанный (уменьшенный) датасет.
# Если размер ~512 px и вес ~50–150 KB — ресайз применён. Если стороны ~4000 px
# и вес в мегабайтах — подключены исходные данные (обучение будет упираться в декод).
sample_path = CFG.TRAIN_IMG_DIR / train_df["image"].iloc[0]
with Image.open(sample_path) as im:
    w, h = im.size
size_kb = sample_path.stat().st_size / 1024
print(f"Источник данных : {CFG.DATA_DIR}")
print(f"Пример          : {sample_path.name}")
print(f"Размер (px)     : {w}x{h}")
print(f"Вес файла       : {size_kb:.0f} KB")
if max(w, h) <= 1024:
    print("OK: данные предварительно уменьшены (ресайз применён).")
else:
    print(
        "ВНИМАНИЕ: похоже, подключены ИСХОДНЫЕ крупные изображения (>1024 px). "
        "Обучение будет медленным. Для скорости подключи 512px-датасет и поправь CFG.DATA_DIR."
    )

In [ ]:
train_df = pd.read_csv(CFG.TRAIN_CSV)
print(f"Train samples: {len(train_df)}")
train_df.head()

In [ ]:
def parse_labels(label_str: str) -> list[str]:
    return label_str.strip().split()


train_df["label_list"] = train_df["labels"].apply(parse_labels)

all_labels = sorted({lbl for labels in train_df["label_list"] for lbl in labels})
label2idx = {lbl: i for i, lbl in enumerate(all_labels)}
idx2label = {i: lbl for lbl, i in label2idx.items()}
num_classes = len(all_labels)

print(f"Classes ({num_classes}): {all_labels}")


def labels_to_multihot(label_list: list[str], n_classes: int) -> np.ndarray:
    vec = np.zeros(n_classes, dtype=np.float32)
    for lbl in label_list:
        vec[label2idx[lbl]] = 1.0
    return vec


train_df["target"] = train_df["label_list"].apply(
    lambda x: labels_to_multihot(x, num_classes)
)

In [ ]:
train_idx, val_idx = train_test_split(
    np.arange(len(train_df)),
    test_size=CFG.VAL_RATIO,
    random_state=CFG.SEED,
    shuffle=True,
)

train_split = train_df.iloc[train_idx].reset_index(drop=True)
val_split = train_df.iloc[val_idx].reset_index(drop=True)
print(f"Train: {len(train_split)}, Val: {len(val_split)}")

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((CFG.IMG_SIZE, CFG.IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((CFG.IMG_SIZE, CFG.IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [ ]:
def load_image_fast(path: Path, size: int) -> Image.Image:
    """Декод JPEG в уменьшенном масштабе.

    Image.draft() просит libjpeg распаковывать в масштабе 1/2, 1/4 или 1/8
    прямо во время декода. Для огромных снимков (~4000 px) это в разы дешевле,
    чем декодировать полное разрешение и потом ресайзить. Узкое место именно тут.
    """
    image = Image.open(path)
    image.draft("RGB", (size, size))  # libjpeg выберет ближайший масштаб ≥ size
    return image.convert("RGB")


class PlantPathologyDataset(Dataset):
    """Хранит только имена файлов и numpy-массив таргетов.

    Важно: НЕ держим pandas.DataFrame с object-колонками. При num_workers > 0
    обращение к Python-объектам через refcount/copy-on-write раздувает RAM
    воркеров вплоть до OOM. Лёгкие numpy/строки этого избегают.
    """

    def __init__(self, df: pd.DataFrame, img_dir: Path, transform=None, img_size: int = 256):
        self.image_names = df["image"].to_numpy()            # массив строк
        self.targets = np.stack(df["target"].to_numpy())     # (N, num_classes) float32
        self.img_dir = img_dir
        self.transform = transform
        self.img_size = img_size

    def __len__(self) -> int:
        return len(self.image_names)

    def __getitem__(self, idx: int):
        img_path = self.img_dir / self.image_names[idx]
        image = load_image_fast(img_path, self.img_size)
        if self.transform:
            image = self.transform(image)
        target = torch.from_numpy(self.targets[idx])
        return image, target


class TestDataset(Dataset):
    def __init__(self, img_dir: Path, transform=None, img_size: int = 256):
        self.img_dir = img_dir
        self.image_names = sorted(
            [p.name for p in img_dir.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"}]
        )
        self.transform = transform
        self.img_size = img_size

    def __len__(self) -> int:
        return len(self.image_names)

    def __getitem__(self, idx: int):
        name = self.image_names[idx]
        image = load_image_fast(self.img_dir / name, self.img_size)
        if self.transform:
            image = self.transform(image)
        return image, name


train_ds = PlantPathologyDataset(train_split, CFG.TRAIN_IMG_DIR, train_transform, CFG.IMG_SIZE)
val_ds = PlantPathologyDataset(val_split, CFG.TRAIN_IMG_DIR, val_transform, CFG.IMG_SIZE)

train_loader = DataLoader(
    train_ds,
    batch_size=CFG.BATCH_SIZE,
    shuffle=True,
    num_workers=CFG.NUM_WORKERS,
    pin_memory=True,
    drop_last=True,
    persistent_workers=CFG.NUM_WORKERS > 0,
    prefetch_factor=2 if CFG.NUM_WORKERS > 0 else None,
)
val_loader = DataLoader(
    val_ds,
    batch_size=CFG.BATCH_SIZE,
    shuffle=False,
    num_workers=CFG.NUM_WORKERS,
    pin_memory=True,
    persistent_workers=CFG.NUM_WORKERS > 0,
    prefetch_factor=2 if CFG.NUM_WORKERS > 0 else None,
)

### Подготовка весов (один раз, Internet: **On**)

Выполните следующую ячейку **один раз** с включённым интернетом, затем сохраните файл из `/kaggle/working/weights/` как Kaggle Dataset и подключите к ноутбуку. После этого можно работать с **Internet: Off**.

In [ ]:
# @title Скачать веса (только при Internet: On)
from torch.hub import download_url_to_file

WEIGHT_URLS = {
    "efficientnet_b0": "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth",
    "resnet50": "https://download.pytorch.org/models/resnet50-0676ba61.pth",
}

prep_dir = CFG.OUTPUT_DIR / "weights"
prep_dir.mkdir(parents=True, exist_ok=True)
dst = prep_dir / BACKBONE_WEIGHT_FILES[CFG.BACKBONE]

if dst.exists():
    print(f"Already exists: {dst}")
elif resolve_weights_path(CFG.BACKBONE) is not None:
    print(f"Weights already available: {resolve_weights_path(CFG.BACKBONE)}")
else:
    try:
        print(f"Downloading {CFG.BACKBONE}...")
        download_url_to_file(WEIGHT_URLS[CFG.BACKBONE], str(dst), progress=True)
        print(f"Saved: {dst}")
        print("→ New Dataset → Upload → добавьте этот .pth → Add Data в ноутбук")
    except Exception as exc:
        print(f"Download skipped ({exc})")
        print("Подключите Kaggle Dataset с файлом:", BACKBONE_WEIGHT_FILES[CFG.BACKBONE])

In [ ]:
def load_backbone(backbone_name: str, weights_path: Path | None):
    if backbone_name == "efficientnet_b0":
        base = models.efficientnet_b0(weights=None)
        in_features = base.classifier[1].in_features
        if weights_path is not None:
            state = torch.load(weights_path, map_location="cpu", weights_only=True)
            base.load_state_dict(state)
        base.classifier = nn.Identity()
        return base, in_features

    if backbone_name == "resnet50":
        base = models.resnet50(weights=None)
        in_features = base.fc.in_features
        if weights_path is not None:
            state = torch.load(weights_path, map_location="cpu", weights_only=True)
            base.load_state_dict(state)
        base.fc = nn.Identity()
        return base, in_features

    raise ValueError(f"Unknown backbone: {backbone_name}")


class MultiLabelCNN(nn.Module):
    """Backbone + linear head; sigmoid применяется в loss/inference."""

    def __init__(self, backbone_name: str, num_classes: int, weights_path: Path | None):
        super().__init__()
        self.backbone, in_features = load_backbone(backbone_name, weights_path)
        self.head = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(in_features, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = self.backbone(x)
        return self.head(features)


weights_path = resolve_weights_path(CFG.BACKBONE)
if weights_path is None:
    raise RuntimeError(
        f"Не найдены веса для {CFG.BACKBONE}.\n"
        f"Ожидаемый файл: {BACKBONE_WEIGHT_FILES[CFG.BACKBONE]}\n"
        "Варианты:\n"
        "  1) Add Data → Kaggle Dataset с этим .pth (путь: CFG.WEIGHTS_DIR)\n"
        "  2) Один раз включить Internet, выполнить ячейку «Подготовка весов», "
        "сохранить .pth как Dataset\n"
        "Скачивание через weights='DEFAULT' при Internet: Off не работает."
    )
print(f"Pretrained weights: {weights_path}")

model = MultiLabelCNN(CFG.BACKBONE, num_classes, weights_path)
model = model.to(device)

if n_gpus > 1:
    model = nn.DataParallel(model)
    print(f"DataParallel: {n_gpus} GPU")
else:
    print("DataParallel не используется — одна GPU")

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG.NUM_EPOCHS)
scaler = GradScaler("cuda", enabled=True)

In [ ]:
def competition_f1(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Mean F1 по каждому изображению (как в соревновании)."""
    f1_scores = []
    for yt, yp in zip(y_true, y_pred):
        f1_scores.append(f1_score(yt, yp, average="macro", zero_division=0))
    return float(np.mean(f1_scores))


@torch.no_grad()
def validate(model, loader, threshold=0.5):
    model.eval()
    all_probs, all_targets = [], []

    for images, targets in tqdm(loader, desc="Val", leave=False):
        images = images.to(device, non_blocking=True)
        with autocast("cuda"):
            logits = model(images)
        probs = torch.sigmoid(logits).float().cpu().numpy()
        all_probs.append(probs)
        all_targets.append(targets.numpy())

    probs = np.vstack(all_probs)
    targets = np.vstack(all_targets)
    preds = (probs >= threshold).astype(int)

    mean_f1 = competition_f1(targets, preds)
    return mean_f1, probs


def train_one_epoch(model, loader, optimizer, criterion, scaler):
    model.train()
    running_loss = 0.0

    for images, targets in tqdm(loader, desc="Train", leave=False):
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast("cuda"):
            logits = model(images)
            loss = criterion(logits, targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)

    return running_loss / len(loader.dataset)

In [ ]:
best_f1 = 0.0
best_state = None
history = []

for epoch in range(1, CFG.NUM_EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, scaler)
    val_f1, _ = validate(model, val_loader, threshold=CFG.THRESHOLD)
    scheduler.step()

    history.append({"epoch": epoch, "train_loss": train_loss, "val_f1": val_f1})
    print(f"Epoch {epoch}/{CFG.NUM_EPOCHS} | loss={train_loss:.4f} | val F1={val_f1:.4f}")

    if val_f1 > best_f1:
        best_f1 = val_f1
        state_dict = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
        best_state = {k: v.cpu().clone() for k, v in state_dict.items()}
        print(f"  -> new best F1: {best_f1:.4f}")

print(f"\nBest validation F1: {best_f1:.4f}")

In [ ]:
if best_state is not None:
    if isinstance(model, nn.DataParallel):
        model.module.load_state_dict(best_state)
    else:
        model.load_state_dict(best_state)

cfg_dict = {
    k: (str(v) if isinstance(v, Path) else v)
    for k, v in vars(CFG).items()
    if not k.startswith("_") and not callable(v)
}

ckpt_path = CFG.OUTPUT_DIR / "best_model.pt"
torch.save(
    {
        "model_state_dict": best_state,
        "label2idx": label2idx,
        "idx2label": idx2label,
        "cfg": cfg_dict,
        "best_f1": best_f1,
    },
    ckpt_path,
)
print(f"Checkpoint saved: {ckpt_path}")

In [ ]:
def probs_to_label_string(probs: np.ndarray, threshold: float) -> str:
    indices = np.where(probs >= threshold)[0]
    if len(indices) == 0:
        indices = [int(probs.argmax())]
    labels = [idx2label[i] for i in indices]
    return " ".join(labels)


@torch.no_grad()
def predict_test(model, loader, threshold=0.5):
    model.eval()
    names, label_strings = [], []

    for images, batch_names in tqdm(loader, desc="Predict test"):
        images = images.to(device, non_blocking=True)
        logits = model(images)
        probs = torch.sigmoid(logits).cpu().numpy()

        for prob, name in zip(probs, batch_names):
            names.append(name)
            label_strings.append(probs_to_label_string(prob, threshold))

    return names, label_strings

In [ ]:
test_ds = TestDataset(CFG.TEST_IMG_DIR, val_transform, CFG.IMG_SIZE)
test_loader = DataLoader(
    test_ds,
    batch_size=CFG.BATCH_SIZE,
    shuffle=False,
    num_workers=CFG.NUM_WORKERS,
    pin_memory=True,
)

print(f"Test images: {len(test_ds)}")

test_names, test_labels = predict_test(model, test_loader, threshold=CFG.THRESHOLD)

submission = pd.DataFrame({"image": test_names, "labels": test_labels})
submission = submission.sort_values("image").reset_index(drop=True)

sub_path = CFG.OUTPUT_DIR / "submission.csv"
submission.to_csv(sub_path, index=False)
print(f"Submission saved: {sub_path}")
submission.head(10)